Homework: median salary gap between Female/Male per country, age > 30.
Signed difference (Female - Male), not absolute - direction matters here.

Pipeline:
- filter (Age > 30)
- -> groupBy(Country, Gender).agg(median_salary, count)
- -> pivot(Gender)                                        # put Female/Male into one row per country
- -> filter(Female is not null AND Male is not null)     # drop countries missing one gender
- -> gap = Female - Male                                  # <-- THE TRANSFORMATION, 3 ways: native / udf / pandas_udf
- -> window: rank countries by |gap|

In [1]:
#spark.stop()

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("udf_homework").config("spark.sql.adaptive.enabled", "false").getOrCreate()
# .config("spark.sql.adaptive.enabled", "false")  - AQE off - want to see the raw plan
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 10:29:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/13 10:29:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
import time
import numpy as np
import pandas as pd
from pyspark.sql import Window
import pyspark.sql.functions as F
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql.types import (
    DoubleType,
    StringType,
    StructType,
    StructField,
)

df = spark.read.csv("Salary.csv", header=True, inferSchema=True).cache()
df.show(10)

+----+------+---------------+--------------------+-------------------+--------+---------+----------+------+
| Age|Gender|Education Level|           Job Title|Years of Experience|  Salary|  Country|      Race|Senior|
+----+------+---------------+--------------------+-------------------+--------+---------+----------+------+
|32.0|  Male|              1|   Software Engineer|                5.0| 90000.0|       UK|     White|     0|
|28.0|Female|              2|        Data Analyst|                3.0| 65000.0|      USA|  Hispanic|     0|
|45.0|  Male|              3|             Manager|               15.0|150000.0|   Canada|     White|     1|
|36.0|Female|              1|     Sales Associate|                7.0| 60000.0|      USA|  Hispanic|     0|
|52.0|  Male|              2|            Director|               20.0|200000.0|      USA|     Asian|     0|
|29.0|  Male|              1|   Marketing Analyst|                2.0| 55000.0|      USA|  Hispanic|     0|
|42.0|Female|              2

In [4]:
# BEFORE the transformation (same for all 3 variants)
def build_wide_table():
    long_table = (
        df.where(F.col("Age") > 30)
        .groupBy("Country", "Gender")
        .agg(
            F.percentile_approx("Salary", 0.5).alias("median_salary"),
            F.count("*").alias("cnt"),
        )
    )
    wide = (
        long_table.groupBy("Country")
        .pivot("Gender", ["Female", "Male"])
        .agg(F.first("median_salary"))
        .where(F.col("Female").isNotNull() & F.col("Male").isNotNull())
    )
    return wide

 
GAP_SCHEMA = StructType(
    [
        StructField("gap", DoubleType()),
        StructField("gap_category", StringType()),
    ]
)


# 1. Native
def add_gap_native(female_col, male_col):
    gap = female_col - male_col
    x = F.abs(gap)
    category = (
        F.when(x >= 20000, "huge")
        .when(x > 10000, "average")
        .when(x > 0, "small")
        .otherwise("none")
    )
    return F.struct(gap.alias("gap"), category.alias("gap_category"))
 
 
# 2. Python UDF
@udf(GAP_SCHEMA)
def add_gap_udf(female, male):
    gap = female - male
    x = abs(gap)
    if x >= 20000:
        category = "huge"
    elif x > 10000:
        category = "average"
    elif x > 0:
        category = "small"
    else:
        category = "none"
    return (gap, category)
 
 
# 3. pandas_udf
@pandas_udf(GAP_SCHEMA)
def add_gap_pandas(female: pd.Series, male: pd.Series) -> pd.DataFrame:
    gap = female - male
    x = gap.abs()
    category = np.select(
        [x >= 20000, x > 10000, x > 0],
        ["huge", "average", "small"],
        default="none",
    )
    return pd.DataFrame({"gap": gap, "gap_category": category})
 
 
# everything AFTER the transformation: filter -> window
def build_pipeline(add_fn):
    # rank by the SIZE of the gap regardless of direction, so we still see
    # "biggest gap" countries first even when some gaps are negative
    w = Window.orderBy(F.desc(F.abs(F.col("gap"))))
    return (
        build_wide_table()
        .withColumn("_gap_struct", add_fn(F.col("Female"), F.col("Male")))  # <- the transformation itself
        .withColumn("gap", F.col("_gap_struct.gap"))
        .withColumn("gap_category", F.col("_gap_struct.gap_category"))
        .drop("_gap_struct")
        .withColumn("rank_by_gap", F.rank().over(w))  # window function
        .orderBy("rank_by_gap")
    )


def run_and_measure(add_fn, label):
    result = build_pipeline(add_fn)
    #print(f"\n{'=' * 80}\n{label}: EXECUTION PLAN (formatted)\n{'=' * 80}")
    #result.explain(mode="formatted")
    t0 = time.time()
    rows = result.collect()
    #result.write.csv(f"result_{label}.csv", header=True, mode="overwrite")
    elapsed = time.time() - t0
    #print(f"\n{label}: collected {len(rows)} rows in {elapsed:.3f}s")
    #for r in rows:
    #    print(r)
    return elapsed

In [5]:
timings = {}
timings["native"] = run_and_measure(add_gap_native, "NATIVE")

26/09/13 10:29:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [6]:
timings["python udf"] = run_and_measure(add_gap_udf, "PYTHON UDF")

26/09/13 10:29:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [7]:
timings["pandas_udf"] = run_and_measure(add_gap_pandas, "PANDAS UDF")

26/09/13 10:29:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [8]:
print(f"\n{'=' * 80}\nSUMMARY\n{'=' * 80}")
for k, v in timings.items():
    print(f"{k:15s}: {v if v is None else f'{v:.3f}s'}")


SUMMARY
native         : 2.054s
python udf     : 3.241s
pandas_udf     : 3.399s
